In [2]:
# -*- coding: UTF-8 -*-
import pandas as pd
import numpy as np
import warnings
from datetime import datetime
warnings.filterwarnings("ignore")
import pymysql
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, DoubleType

# 初始化SparkSession
spark = SparkSession.builder \
    .appName("SugarDataProcess") \
    .enableHiveSupport() \
    .getOrCreate()

# MySQL配置
SOURCE_CONFIG = {
    "host": "172.20.16.9", "port": 3306,
    "user": "etl_user", "password": "Easugar@123",
    "database": "lims_gxdyty", "charset": "utf8"
}

# 字符串字段定义
STRING_FIELDS = [
    'source_id', 'factory_code', 'season_year', 'category', 'operating_period'
]

def get_data_from_mysql(season_cs):
    """从MySQL读取榨季数据"""
    try:
        conn = pymysql.connect(**SOURCE_CONFIG)
        cursor = conn.cursor()
        
        sql = """
SELECT 
    a.id,a.org_code as '公司名称',a.crushing_season,a.category,
    MAX(CASE WHEN itemname = '开机—停机时间' THEN itemvalue END) AS 开机—停机时间,
    MAX(CASE WHEN itemname = '炼糖经过天数' THEN itemvalue END) AS 炼糖经过天数,
    MAX(CASE WHEN itemname = '实际炼糖天数' THEN itemvalue END) AS 实际炼糖天数,
    MAX(CASE WHEN itemname = '溶糖量' THEN itemvalue END) AS 溶糖量,
    MAX(CASE WHEN itemname = '洄溶原糖量' THEN itemvalue END) AS 洄溶原糖量,
    MAX(CASE WHEN itemname = '洄溶等外糖量（含糖粉）' THEN itemvalue END) AS 洄溶等外糖量（含糖粉）,
    MAX(CASE WHEN itemname = '洄溶其它糖量' THEN itemvalue END) AS 洄溶其它糖量,
    MAX(CASE WHEN itemname = '洄溶原糖色值' THEN IFNULL(itemvalue,0) END) AS 洄溶原糖色值,
    MAX(CASE WHEN itemname = '洄溶原糖蔗糖分' THEN itemvalue END) AS 洄溶原糖蔗糖分,
    MAX(CASE WHEN itemname = '原糖入库量' THEN itemvalue END) AS 原糖入库量,
    MAX(CASE WHEN itemname = '实际炼糖时间' THEN itemvalue END) AS 实际炼糖时间,
    MAX(CASE WHEN itemname = '设备故障损失时间' THEN itemvalue END) AS 设备故障损失时间,
    MAX(CASE WHEN itemname = '小期检修时间' THEN itemvalue END) AS 小期检修时间,
    MAX(CASE WHEN itemname = '其他损失时间' THEN itemvalue END) AS 其他损失时间,
    MAX(CASE WHEN itemname = '设备安全率' THEN itemvalue END) AS 设备安全率,
    MAX(CASE WHEN itemname = '特级精制白砂糖' THEN itemvalue END) AS 特级精制白砂糖,
    MAX(CASE WHEN itemname = '精制白砂糖' THEN itemvalue END) AS 精制白砂糖,
    MAX(CASE WHEN itemname = '精幼白砂糖' THEN itemvalue END) AS 精幼白砂糖,
    MAX(CASE WHEN itemname = '优级白砂糖' THEN itemvalue END) AS 优级白砂糖,
    MAX(CASE WHEN itemname = '优幼白砂糖' THEN itemvalue END) AS 优幼白砂糖,
    MAX(CASE WHEN itemname = 'CA白砂糖' THEN itemvalue END) AS CA白砂糖,
    MAX(CASE WHEN itemname = 'C1白砂糖' THEN itemvalue END) AS C1白砂糖,
    MAX(CASE WHEN itemname = '入库等外白砂糖（含糖粉）' THEN itemvalue END) AS 入库等外白砂糖（含糖粉）,
    MAX(CASE WHEN itemname = '精优级品率' THEN itemvalue END) AS 精优级品率,
    MAX(CASE WHEN itemname = '混合糖产量' THEN itemvalue END) AS 混合糖产量,
    MAX(CASE WHEN itemname = '回原线R3糖蜜重量' THEN itemvalue END) AS 回原线R3糖蜜重量,
    MAX(CASE WHEN itemname = '开榨期产生的陈化仓白砂糖量' THEN itemvalue END) AS 开榨期产生的陈化仓白砂糖量,
    MAX(CASE WHEN itemname = '陈化仓白砂糖量' THEN itemvalue END) AS 陈化仓白砂糖量,
    MAX(CASE WHEN itemname = '在制品等折白砂糖量' THEN itemvalue END) AS 在制品等折白砂糖量,
    MAX(CASE WHEN itemname = '产糖率' THEN itemvalue END) AS 产糖率,
    MAX(CASE WHEN itemname = '产糖率(含糖蜜等折量)' THEN itemvalue END) AS 产糖率（含糖蜜等折量）,
    MAX(CASE WHEN itemname = '产糖率(含在制品等折量) ' THEN itemvalue END) AS 产糖率（含在制品等折量）,
    MAX(CASE WHEN itemname = '氧化钙' THEN itemvalue END) AS 氧化钙,
    MAX(CASE WHEN itemname = '助滤剂' THEN itemvalue END) AS 助滤剂,
    MAX(CASE WHEN itemname = '耗电量' THEN itemvalue END) AS 耗电量,
    MAX(CASE WHEN itemname = '耗汽量' THEN itemvalue END) AS 耗汽量,
    MAX(CASE WHEN itemname = '吨洄溶糖耗电量' THEN itemvalue END) AS 吨洄溶糖耗电量,
    MAX(CASE WHEN itemname = '吨洄溶糖耗蒸汽量' THEN itemvalue END) AS 吨洄溶糖耗蒸汽量,
    MAX(CASE WHEN itemname = '原料糖蔗糖量' THEN itemvalue END) AS 原料糖蔗糖量,
    MAX(CASE WHEN itemname = '废蜜产量 (含R3蜜等折量)' THEN itemvalue END) AS 废蜜产量（含R3蜜等折量）,
    MAX(CASE WHEN itemname = '废蜜产率(含R3蜜等折量)' THEN itemvalue END) AS 废蜜产率（含R3蜜等折量）,
    MAX(CASE WHEN itemname = '滤泥损失' THEN itemvalue END) AS 滤泥损失,
    MAX(CASE WHEN itemname = '废蜜损失' THEN itemvalue END) AS 废蜜损失,
    MAX(CASE WHEN itemname = '不可测定损失' THEN itemvalue END) AS 不可测定损失,
    MAX(CASE WHEN itemname = '总收回率' THEN itemvalue END) AS 总收回率,
    MAX(CASE WHEN itemname = '总收回率(含糖蜜等折量)' THEN itemvalue END) AS 总收回率（含糖蜜等折量）,
    MAX(CASE WHEN itemname = '供外电量' THEN itemvalue END) AS 供外电量,
    MAX(CASE WHEN itemname = '供其他电量' THEN itemvalue END) AS 供其他电量   
FROM lcgl_search_plan_b a 
inner join lcgl_date_b b on a.org_code=CONVERT(b.org_code USING gbk)  and a.crushing_season=CONVERT(b.name USING gbk) 
where a.org_code in('FNR','CZ','NMR','HT') and  a.category='炼糖生产'
AND b.code= %s
group by a.org_code,a.crushing_season,a.category    
ORDER BY
    CASE
        WHEN  a.org_code = 'FNR' THEN 1
        WHEN  a.org_code = 'CZ' THEN 2
        WHEN  a.org_code = 'NMR' THEN 3
        ELSE 4
    END
"""
        cursor.execute(sql, (season_cs,))
        result = cursor.fetchall()
        cols = [c[0] for c in cursor.description]
        df = pd.DataFrame(result, columns=cols) if result else pd.DataFrame(columns=cols)
        return df
        
    except Exception as e:
        print(f"获取数据失败: {e}")
        raise
    finally:
        if cursor: cursor.close()
        if conn: conn.close()

def translate_column_names(df):
    """字段名翻译+添加注释"""
    column_mapping = {
        '公司名称': ('factory_code', '工厂代码'),
        'crushing_season': ('season_year', '榨季年份'),
        'category': ('category', '类别'),
        '开机—停机时间': ('operating_period', '开机—停机时间'),
        '炼糖经过天数': ('sugar_refining_days', '炼糖经过天数'),
        '实际炼糖天数': ('actual_refining_days', '实际炼糖天数'),
        '溶糖量': ('dissolved_sugar_amount', '溶糖量'),
        '洄溶原糖量': ('recirculated_raw_sugar_amount', '洄溶原糖量'),
        '洄溶等外糖量（含糖粉）': ('recirculated_off_grade_sugar_amount', '洄溶等外糖量（含糖粉）'),
        '洄溶其它糖量': ('recirculated_other_sugar_amount', '洄溶其它糖量'),
        '洄溶原糖色值': ('recirculated_raw_sugar_color', '洄溶原糖色值'),
        '洄溶原糖蔗糖分': ('recirculated_raw_sugar_sucrose', '洄溶原糖蔗糖分'),
        '原糖入库量': ('raw_sugar_inventory', '原糖入库量'),
        '实际炼糖时间': ('actual_refining_time', '实际炼糖时间'),
        '设备故障损失时间': ('equipment_failure_loss_time', '设备故障损失时间'),
        '小期检修时间': ('minor_maintenance_time', '小期检修时间'),
        '其他损失时间': ('other_loss_time', '其他损失时间'),
        '设备安全率': ('equipment_safety_rate', '设备安全率'),
        '特级精制白砂糖': ('special_refined_sugar', '特级精制白砂糖'),
        '精制白砂糖': ('refined_sugar', '精制白砂糖'),
        '精幼白砂糖': ('refined_young_sugar', '精幼白砂糖'),
        '优级白砂糖': ('superior_sugar', '优级白砂糖'),
        '优幼白砂糖': ('superior_young_sugar', '优幼白砂糖'),
        'CA白砂糖': ('ca_sugar', 'CA白砂糖'),
        'C1白砂糖': ('c1_sugar', 'C1白砂糖'),
        '入库等外白砂糖（含糖粉）': ('stored_off_grade_sugar', '入库等外白砂糖（含糖粉）'),
        '精优级品率': ('fine_superior_rate', '精优级品率'),
        '混合糖产量': ('mixed_sugar_production', '混合糖产量'),
        '回原线R3糖蜜重量': ('r3_molasses_weight_to_original', '回原线R3糖蜜重量'),
        '开榨期产生的陈化仓白砂糖量': ('aged_white_sugar_during_crushing', '开榨期产生的陈化仓白砂糖量'),
        '陈化仓白砂糖量': ('aged_white_sugar', '陈化仓白砂糖量'),
        '在制品等折白砂糖量': ('wip_equivalent_white_sugar', '在制品等折白砂糖量'),
        '产糖率': ('sugar_yield_rate', '产糖率'),
        '产糖率（含糖蜜等折量）': ('sugar_yield_rate_with_molasses', '产糖率（含糖蜜等折量）'),
        '产糖率（含在制品等折量）': ('sugar_yield_rate_with_wip', '产糖率（含在制品等折量）'),
        '氧化钙': ('calcium_oxide', '氧化钙'),
        '助滤剂': ('filter_aid', '助滤剂'),
        '耗电量': ('power_consumption', '耗电量'),
        '耗汽量': ('steam_consumption', '耗汽量'),
        '吨洄溶糖耗电量': ('power_consumption_per_ton_recirculated_sugar', '吨洄溶糖耗电量'),
        '吨洄溶糖耗蒸汽量': ('steam_consumption_per_ton_recirculated_sugar', '吨洄溶糖耗蒸汽量'),
        '原料糖蔗糖量': ('raw_sugar_sucrose_amount', '原料糖蔗糖量'),
        '废蜜产量（含R3蜜等折量）': ('molasses_production_with_r3', '废蜜产量（含R3蜜等折量）'),
        '废蜜产率（含R3蜜等折量）': ('molasses_yield_rate_with_r3', '废蜜产率（含R3蜜等折量）'),
        '滤泥损失': ('filter_mud_loss', '滤泥损失'),
        '废蜜损失': ('molasses_loss', '废蜜损失'),
        '不可测定损失': ('undetermined_loss', '不可测定损失'),
        '总收回率': ('total_recovery_rate', '总收回率'),
        '总收回率（含糖蜜等折量）': ('total_recovery_rate_with_molasses', '总收回率（含糖蜜等折量）'),
        '供外电量': ('external_power_supply', '供外电量'),
        '供其他电量': ('other_power_supply', '供其他电量'),
        'id': ('source_id', '源数据ID')
    }
    
    df_renamed = df.rename(columns={ch: en for ch, (en, _) in column_mapping.items() if ch in df.columns})
    column_comments = {en: comment for ch, (en, comment) in column_mapping.items() if ch in df.columns}
    return df_renamed, column_comments

def convert_column_types(pandas_df):
    """字段类型转换"""
    for col in pandas_df.columns:
        if col in STRING_FIELDS:
            pandas_df[col] = pandas_df[col].fillna('').astype(str)
        else:
            pandas_df[col] = pd.to_numeric(pandas_df[col], errors='coerce').fillna(0.0).astype(float)
    return pandas_df

def create_spark_df(pandas_df):
    """Pandas转Spark DataFrame"""
    pandas_df = convert_column_types(pandas_df)
    return spark.createDataFrame(pandas_df)

# ===================== 核心主逻辑：合并数据 + 无分区覆盖写入 =====================
if __name__ == "__main__":
    TARGET_TABLE = "app.app_lims_liantang_target_f_1d"
    SEASON_LIST = ["2024", "2025", "2026"]
    merged_df = None
    final_column_comments = None

    for season in SEASON_LIST:
        try:
            print(f"\n处理榨季: {season}")
            pandas_df = get_data_from_mysql(season)
            if pandas_df.empty:
                continue
            pandas_df_renamed, column_comments = translate_column_names(pandas_df)
            if final_column_comments is None:
                final_column_comments = column_comments
            spark_df = create_spark_df(pandas_df_renamed)
            merged_df = spark_df if merged_df is None else merged_df.union(spark_df)
            print(f"榨季 {season} 已合并")
        except Exception as e:
            print(f"榨季 {season} 失败: {e}")
            continue

    if merged_df is not None:
        # 写入表（无分区，覆盖）
        merged_df.write.mode("overwrite").saveAsTable(TARGET_TABLE)
        print(f"表 {TARGET_TABLE} 写入完成，共 {merged_df.count()} 行")

        # 添加字段注释（必须带数据类型）
        if final_column_comments:
            # 从 DataFrame schema 获取字段名→数据类型映射
            from pyspark.sql.types import StringType, DoubleType

            def spark_type_to_sql(dt):
                if isinstance(dt, StringType):
                    return "STRING"
                elif isinstance(dt, DoubleType):
                    return "DOUBLE"
                else:
                    # 兜底：返回 Spark 类型的简单大写名（如 INTEGER, DECIMAL(10,2)）
                    return dt.simpleString().upper()

            col_type_map = {field.name: spark_type_to_sql(field.dataType) 
                            for field in merged_df.schema.fields}

            for col_name, comment in final_column_comments.items():
                if col_name not in col_type_map:
                    print(f"警告：列 {col_name} 不存在于 DataFrame 中，跳过注释")
                    continue
                col_type = col_type_map[col_name]
                # 注意：列名用反引号包裹，防止特殊字符
                alter_sql = f"""
                    ALTER TABLE {TARGET_TABLE} 
                    CHANGE COLUMN `{col_name}` `{col_name}` {col_type} 
                    COMMENT '{comment}'
                """
                try:
                    spark.sql(alter_sql)
                    print(f"✅ 注释添加成功: {col_name} -> {comment}")
                except Exception as e:
                    print(f"❌ 注释添加失败 {col_name}: {e}")
        else:
            print("无注释信息可添加")
    else:
        print("无有效数据可写入")

    spark.stop()

/opt/tiger/spark/python/pyspark/context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning



处理榨季: 2024
榨季 2024 已合并

处理榨季: 2025
榨季 2025 已合并

处理榨季: 2026
榨季 2026 已合并
表 app.app_lims_liantang_target_f_1d 写入完成，共 12 行
✅ 注释添加成功: factory_code -> 工厂代码
✅ 注释添加成功: season_year -> 榨季年份
✅ 注释添加成功: category -> 类别
✅ 注释添加成功: operating_period -> 开机—停机时间
✅ 注释添加成功: sugar_refining_days -> 炼糖经过天数
✅ 注释添加成功: actual_refining_days -> 实际炼糖天数
✅ 注释添加成功: dissolved_sugar_amount -> 溶糖量
✅ 注释添加成功: recirculated_raw_sugar_amount -> 洄溶原糖量
✅ 注释添加成功: recirculated_off_grade_sugar_amount -> 洄溶等外糖量（含糖粉）
✅ 注释添加成功: recirculated_other_sugar_amount -> 洄溶其它糖量
✅ 注释添加成功: recirculated_raw_sugar_color -> 洄溶原糖色值
✅ 注释添加成功: recirculated_raw_sugar_sucrose -> 洄溶原糖蔗糖分
✅ 注释添加成功: raw_sugar_inventory -> 原糖入库量
✅ 注释添加成功: actual_refining_time -> 实际炼糖时间
✅ 注释添加成功: equipment_failure_loss_time -> 设备故障损失时间
✅ 注释添加成功: minor_maintenance_time -> 小期检修时间
✅ 注释添加成功: other_loss_time -> 其他损失时间
✅ 注释添加成功: equipment_safety_rate -> 设备安全率
✅ 注释添加成功: special_refined_sugar -> 特级精制白砂糖
✅ 注释添加成功: refined_sugar -> 精制白砂糖
✅ 注释添加成功: refined_young_sugar -> 精幼白砂糖
✅ 注释添加成